# Бінарна класифікація: прогнозування дефолту (`gb`)

**Ціль:** Побудувати модель бінарної класифікації для прогнозування цільової змінної `gb` (0 = немає дефолту, 1 = дефолт).

**Дані:** `train_df.csv` — ~26,824 рядки × 554 колонки.
- `id` — ідентифікатор об'єкта
- `gb` — цільова змінна (0 або 1)
- `cat_*` — категоріальні ознаки
- `num_*` — числові ознаки

**Алгоритми:**
1. Logistic Regression (обов'язково)
2. CatBoost (на основі дерев рішень)
3. LightGBM (на основі дерев рішень)

---

## Структура ноутбука

1. [Імпорти та конфігурація](#1)
2. [Завантаження даних та початковий огляд (EDA)](#2)
3. [Аналіз пропущених значень](#3)
4. [Розподіл ознак та статистичні тести](#4)
5. [Мультиваріатний аналіз](#5)
6. [Очистка та відбір ознак](#6)
7. [Розбивка на train/val та препроцесинг](#7)
8. [Logistic Regression](#8)
9. [CatBoost](#9)
10. [LightGBM](#10)
11. [Порівняння моделей та фінальні графіки](#11)

## 1. Імпорти та конфігурація

Завантажуємо всі необхідні бібліотеки. Всі утиліти визначені прямо в цьому ноутбуці — жодних зовнішніх `.py` файлів не потрібно.

In [1]:
import warnings
import logging
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from scipy.stats import chi2_contingency, mannwhitneyu

from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    f1_score,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
    confusion_matrix,
    ConfusionMatrixDisplay,
)
from sklearn.calibration import calibration_curve
from sklearn.isotonic import IsotonicRegression

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.WARNING)

# ─── Константи ────────────────────────────────────────────────────────────────
DATA_PATH    = "train_df.csv"
TARGET_COL   = "gb"
ID_COL       = "id"
RANDOM_STATE = 42
N_SPLITS     = 5

# Корпоративна палітра кольорів для графіків
PALETTE = {
    "primary":   "#2E86AB",
    "secondary": "#A23B72",
    "accent":    "#F18F01",
    "pos":       "#C73E1D",
    "neg":       "#3A7D44",
}

# Стиль matplotlib
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette([PALETTE["primary"], PALETTE["secondary"], PALETTE["accent"]])
plt.rcParams.update({"figure.dpi": 100, "axes.titlesize": 13, "axes.labelsize": 11})
%matplotlib inline

np.random.seed(RANDOM_STATE)
print("✓ Імпорти та конфігурація завантажені")

✓ Імпорти та конфігурація завантажені


## 2. Завантаження даних та початковий огляд (EDA)

Завантажуємо датасет і проводимо базовий аналіз:
- Розмір та структура
- Баланс класів (цільова змінна)
- Загальна статистика

## 3. Аналіз пропущених значень

Досліджуємо структуру пропусків:
- Які колонки повністю порожні?
- Чи залежить факт пропуску від цільової змінної? (MNAR — Missing Not At Random)

## 4. Розподіл числових ознак та статистичні тести

Перевіряємо, які ознаки статистично значущо відрізняються між класами (Mann-Whitney U test).
Також аналізуємо skewness та outliers.

## 5. Мультиваріатний аналіз

- Кореляційна матриця топ-20 числових ознак
- Кардинальність категоріальних ознак
- Активність entities (рядків на ID) vs частота дефолту

## 6. Очистка та відбір ознак

Видаляємо:
1. Колонки з 100% пропусків
2. Точні дублікати колонок
3. Quasi-constant ознаки (дисперсія < 0.01)
4. Сильно скорельовані пари числових ознак (|r| > 0.95)

## 7. Розбивка на train/val та препроцесинг

**Стратегія розбивки:** `StratifiedGroupKFold` по `id` — гарантує, що один і той самий `id` не потрапить одночасно в train і val. Це критично для панельних даних.

**Препроцесинг для Logistic Regression:**
- Додаємо бінарні індикатори пропусків для MNAR ознак
- Imputation медіаною (для числових) та модою (для категоріальних)
- RobustScaler (стійкий до outliers)

In [ ]:
# ─── Готуємо дані ───────────────────────────────────────────────────────────

# Беремо тільки відфільтровані ознаки
X_all = df[FINAL_FEATURES].copy()
y_all = df[TARGET_COL].values
groups_all = df[ID_COL].values  # для GroupKFold

# Визначаємо числові та категоріальні колонки в фінальному наборі
final_num = [c for c in FINAL_FEATURES if c.startswith("num_")]
final_cat = [c for c in FINAL_FEATURES if c.startswith("cat_")]

# MNAR ознаки в фінальному наборі
final_mnar = [c for c in MNAR_COLS if c in final_num]

print(f"Фінальних числових ознак:       {len(final_num)}")
print(f"Фінальних категоріальних ознак: {len(final_cat)}")
print(f"MNAR ознак (для індикаторів):   {len(final_mnar)}")

# ─── Stratified Group K-Fold ─────────────────────────────────────────────────
# Для стратифікації потрібна мітка на рівні entity (беремо першу)
entity_target = df.groupby(ID_COL)[TARGET_COL].first().reset_index()
entity_y = df[ID_COL].map(entity_target.set_index(ID_COL)[TARGET_COL]).values

sgkf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

splits = list(sgkf.split(X_all, entity_y, groups=groups_all))

print(f"\nCV splits (StratifiedGroupKFold, {N_SPLITS} folds):")
for i, (tr, va) in enumerate(splits):
    print(f"  Fold {i}: train={len(tr):,}, val={len(va):,}, "
          f"val positive rate={y_all[va].mean():.2%}")

In [ ]:
# ─── Функції препроцесингу ──────────────────────────────────────────────────

def encode_categoricals(X_train, X_val, cat_cols):
    """
    Label encoding категоріальних ознак.
    Навчаємо на train, застосовуємо до val (leakage-safe).
    """
    X_tr, X_va = X_train.copy(), X_val.copy()
    for col in cat_cols:
        if col not in X_tr.columns:
            continue
        uniques = X_tr[col].astype(str).fillna("__NA__").unique()
        mapping = {v: i for i, v in enumerate(sorted(uniques))}
        X_tr[col] = X_tr[col].astype(str).fillna("__NA__").map(mapping).fillna(-1).astype(int)
        X_va[col] = X_va[col].astype(str).fillna("__NA__").map(mapping).fillna(-1).astype(int)
    return X_tr, X_va


def sanitize_names(cols):
    """Замінюємо спецсимволи в назвах колонок (потрібно для LightGBM)."""
    return [re.sub(r"[^\w]", "_", c) for c in cols]


def prepare_for_logreg(X_train, y_train, X_val, mnar_cols, num_cols, cat_cols):
    """
    Повний pipeline препроцесингу для Logistic Regression:
    1. MNAR індикатори (is_missing бінарна ознака)
    2. Label encoding категоріальних
    3. Median imputation числових
    4. Mode imputation категоріальних
    5. RobustScaler
    """
    X_tr = X_train.copy()
    X_va = X_val.copy()

    # 1. MNAR індикатори
    for col in mnar_cols:
        if col in X_tr.columns:
            X_tr[f"{col}__miss"] = X_tr[col].isnull().astype(np.int8)
            X_va[f"{col}__miss"] = X_va[col].isnull().astype(np.int8)

    # 2. Label encoding
    X_tr, X_va = encode_categoricals(X_tr, X_va, cat_cols)

    all_cols = X_tr.columns.tolist()

    # 3. Imputation медіаною (все разом — числові вже закодовані, категоріальні теж int)
    medians = X_tr.median()
    X_tr = X_tr.fillna(medians)
    X_va = X_va.fillna(medians)

    # 4. RobustScaler
    scaler = RobustScaler()
    X_tr_s = pd.DataFrame(scaler.fit_transform(X_tr), columns=all_cols, index=X_tr.index)
    X_va_s = pd.DataFrame(scaler.transform(X_va),    columns=all_cols, index=X_va.index)

    return X_tr_s, X_va_s


def prepare_for_boosting(X_train, X_val, cat_cols):
    """
    Препроцесинг для CatBoost/LightGBM:
    Тільки label encoding категоріальних.
    CatBoost/LightGBM обробляють NaN нативно.
    """
    return encode_categoricals(X_train, X_val, cat_cols)


print("✓ Функції препроцесингу визначені")

## 8. Logistic Regression

**Чому Logistic Regression:**
- Інтерпретована (коефіцієнти = внесок кожної ознаки)
- Швидке навчання
- Хороший baseline

**Проблеми на цих даних:**
- Мультиколінеарність (вирішена видаленням скорельованих ознак)
- Незбалансований датасет (вирішено `class_weight='balanced'`)
- Пропуски (вирішено imputation + MNAR indicators)

In [ ]:
# ─── Logistic Regression: Cross-Validation ──────────────────────────────────

oof_lr = np.full(len(y_all), np.nan)
lr_fold_metrics = []

for fold_idx, (train_idx, val_idx) in enumerate(splits):
    X_tr_raw = X_all.iloc[train_idx]
    X_va_raw = X_all.iloc[val_idx]
    y_tr = y_all[train_idx]
    y_va = y_all[val_idx]

    # Препроцесинг для LR
    X_tr_lr, X_va_lr = prepare_for_logreg(
        X_tr_raw, y_tr, X_va_raw,
        mnar_cols=final_mnar,
        num_cols=final_num,
        cat_cols=final_cat
    )

    # Logistic Regression з L2 regularization та class_weight
    lr = LogisticRegression(
        C=0.1,
        penalty="l2",
        solver="saga",
        max_iter=300,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    lr.fit(X_tr_lr, y_tr)
    val_prob = lr.predict_proba(X_va_lr)[:, 1]
    oof_lr[val_idx] = val_prob

    pr_auc  = average_precision_score(y_va, val_prob)
    roc_auc = roc_auc_score(y_va, val_prob)
    lr_fold_metrics.append({"fold": fold_idx, "pr_auc": pr_auc, "roc_auc": roc_auc})
    print(f"  Fold {fold_idx}: PR-AUC={pr_auc:.4f} | ROC-AUC={roc_auc:.4f}")

lr_metrics_df = pd.DataFrame(lr_fold_metrics)
print(f"\n  Logistic Regression — середнє CV:")
print(f"    PR-AUC:  {lr_metrics_df['pr_auc'].mean():.4f} ± {lr_metrics_df['pr_auc'].std():.4f}")
print(f"    ROC-AUC: {lr_metrics_df['roc_auc'].mean():.4f} ± {lr_metrics_df['roc_auc'].std():.4f}")

## 9. CatBoost

**Чому CatBoost:**
- Нативна підтримка категоріальних ознак
- Обробляє NaN без попереднього заповнення
- Стійкий до outliers та skewness
- `auto_class_weights='Balanced'` — автоматичне зважування класів

In [ ]:
# ─── CatBoost: Cross-Validation ─────────────────────────────────────────────
from catboost import CatBoostClassifier, Pool

oof_cb = np.full(len(y_all), np.nan)
cb_fold_metrics = []

# Індекси категоріальних ознак (CatBoost приймає position у списку)
cat_feature_indices = [FINAL_FEATURES.index(c) for c in final_cat if c in FINAL_FEATURES]

for fold_idx, (train_idx, val_idx) in enumerate(splits):
    X_tr_raw = X_all.iloc[train_idx]
    X_va_raw = X_all.iloc[val_idx]
    y_tr = y_all[train_idx]
    y_va = y_all[val_idx]

    # Для CatBoost: тільки label encoding категоріальних, NaN залишаємо
    X_tr_cb, X_va_cb = prepare_for_boosting(X_tr_raw, X_va_raw, final_cat)

    # Перейменовуємо колонки (CatBoost не любить спецсимволи)
    safe_names = sanitize_names(X_tr_cb.columns.tolist())
    X_tr_cb.columns = safe_names
    X_va_cb.columns = safe_names

    train_pool = Pool(X_tr_cb, y_tr, cat_features=cat_feature_indices)
    val_pool   = Pool(X_va_cb, y_va, cat_features=cat_feature_indices)

    cb = CatBoostClassifier(
        iterations=300,
        learning_rate=0.05,
        depth=6,
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=RANDOM_STATE,
        verbose=0,
        allow_writing_files=False,
        auto_class_weights="Balanced",
    )
    cb.fit(train_pool, eval_set=val_pool, use_best_model=True, early_stopping_rounds=50)

    val_prob = cb.predict_proba(val_pool)[:, 1]
    oof_cb[val_idx] = val_prob

    pr_auc  = average_precision_score(y_va, val_prob)
    roc_auc = roc_auc_score(y_va, val_prob)
    cb_fold_metrics.append({"fold": fold_idx, "pr_auc": pr_auc, "roc_auc": roc_auc})
    print(f"  Fold {fold_idx}: PR-AUC={pr_auc:.4f} | ROC-AUC={roc_auc:.4f}")

cb_metrics_df = pd.DataFrame(cb_fold_metrics)
print(f"\n  CatBoost — середнє CV:")
print(f"    PR-AUC:  {cb_metrics_df['pr_auc'].mean():.4f} ± {cb_metrics_df['pr_auc'].std():.4f}")
print(f"    ROC-AUC: {cb_metrics_df['roc_auc'].mean():.4f} ± {cb_metrics_df['roc_auc'].std():.4f}")

# Зберігаємо назви ознак для важливості
CB_FEATURE_NAMES = safe_names

## 10. LightGBM

**Чому LightGBM:**
- Найшвидший з бустинг-алгоритмів
- Histogram-based — ефективний на широких датасетах
- `scale_pos_weight` — зважування класів для незбалансованих даних

In [ ]:
# ─── LightGBM: Cross-Validation ─────────────────────────────────────────────
import lightgbm as lgb

oof_lgb = np.full(len(y_all), np.nan)
lgb_fold_metrics = []

for fold_idx, (train_idx, val_idx) in enumerate(splits):
    X_tr_raw = X_all.iloc[train_idx]
    X_va_raw = X_all.iloc[val_idx]
    y_tr = y_all[train_idx]
    y_va = y_all[val_idx]

    X_tr_lgb, X_va_lgb = prepare_for_boosting(X_tr_raw, X_va_raw, final_cat)

    # Перейменовуємо (LightGBM теж не любить спецсимволи)
    safe_names_lgb = sanitize_names(X_tr_lgb.columns.tolist())
    X_tr_lgb.columns = safe_names_lgb
    X_va_lgb.columns = safe_names_lgb

    # scale_pos_weight — автоматично враховує дисбаланс
    pos = float(y_tr.sum())
    neg = float(len(y_tr) - pos)
    spw = neg / pos if pos > 0 else 1.0

    cat_names = [sanitize_names([c])[0] for c in final_cat if c in X_tr_lgb.columns]

    lgbm = lgb.LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=63,
        objective="binary",
        metric="average_precision",
        scale_pos_weight=spw,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=-1,
    )
    lgbm.fit(
        X_tr_lgb, y_tr,
        eval_set=[(X_va_lgb, y_va)],
        categorical_feature=cat_names if cat_names else "auto",
        callbacks=[
            lgb.early_stopping(stopping_rounds=50, verbose=False),
            lgb.log_evaluation(period=0),
        ],
    )

    val_prob = lgbm.predict_proba(X_va_lgb)[:, 1]
    oof_lgb[val_idx] = val_prob

    pr_auc  = average_precision_score(y_va, val_prob)
    roc_auc = roc_auc_score(y_va, val_prob)
    lgb_fold_metrics.append({"fold": fold_idx, "pr_auc": pr_auc, "roc_auc": roc_auc})
    print(f"  Fold {fold_idx}: PR-AUC={pr_auc:.4f} | ROC-AUC={roc_auc:.4f}")

lgb_metrics_df = pd.DataFrame(lgb_fold_metrics)
print(f"\n  LightGBM — середнє CV:")
print(f"    PR-AUC:  {lgb_metrics_df['pr_auc'].mean():.4f} ± {lgb_metrics_df['pr_auc'].std():.4f}")
print(f"    ROC-AUC: {lgb_metrics_df['roc_auc'].mean():.4f} ± {lgb_metrics_df['roc_auc'].std():.4f}")

LGB_FEATURE_NAMES = safe_names_lgb

## 11. Порівняння моделей та фінальні графіки

- Таблиця метрик для всіх трьох моделей
- PR-curves (Precision-Recall)
- ROC-curves
- Calibration curves
- F1 vs Threshold
- Feature Importance (CatBoost та LightGBM)
- Confusion Matrix при оптимальному threshold

In [ ]:
# ─── Порівняльна таблиця метрик ─────────────────────────────────────────────

def find_optimal_f1(y_true, y_prob, n=200):
    """Знаходимо threshold, що максимізує F1."""
    thresholds = np.linspace(0.001, 0.999, n)
    best_f1, best_t = 0.0, 0.5
    for t in thresholds:
        f1 = f1_score(y_true, (y_prob >= t).astype(int), zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return best_t, best_f1


models = {
    "Logistic Regression": oof_lr,
    "CatBoost":            oof_cb,
    "LightGBM":            oof_lgb,
}

results = []
for name, oof in models.items():
    valid = ~np.isnan(oof)
    y_v = y_all[valid]
    p_v = oof[valid]

    pr_auc  = average_precision_score(y_v, p_v)
    roc_auc = roc_auc_score(y_v, p_v)
    brier   = brier_score_loss(y_v, p_v)
    opt_t, max_f1 = find_optimal_f1(y_v, p_v)

    results.append({
        "Модель":          name,
        "PR-AUC":          round(pr_auc, 4),
        "ROC-AUC":         round(roc_auc, 4),
        "Brier Score":     round(brier, 4),
        "Оптим. Threshold": round(opt_t, 3),
        "Max F1":          round(max_f1, 4),
    })

results_df = pd.DataFrame(results).set_index("Модель")
print("=" * 65)
print("  ПІДСУМКОВА ТАБЛИЦЯ МЕТРИК (OOF — Out-of-Fold CV)")
print("=" * 65)
display(results_df)
print("""
Пояснення метрик:
  PR-AUC (Average Precision) — ГОЛОВНА метрика для незбалансованих даних.
    Чим вище — тим краще модель знаходить позитивні класи.
  ROC-AUC — якість ранжування (1.0 = ідеальна, 0.5 = випадкова).
  Brier Score — calibration (чим менше — тим краще).
  Max F1 — найкращий F1 при оптимальному threshold.
""")

In [ ]:
# ─── PR-curves та ROC-curves ────────────────────────────────────────────────
colors = [PALETTE["accent"], PALETTE["primary"], PALETTE["secondary"]]

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# PR Curves
for (name, oof), color in zip(models.items(), colors):
    valid = ~np.isnan(oof)
    prec, rec, _ = precision_recall_curve(y_all[valid], oof[valid])
    ap = average_precision_score(y_all[valid], oof[valid])
    axes[0].plot(rec, prec, label=f"{name} (AP={ap:.4f})", color=color, lw=2)

# Baseline (random classifier)
baseline = y_all.mean()
axes[0].axhline(baseline, color="gray", ls="--", lw=1, label=f"Baseline ({baseline:.3f})")
axes[0].set_xlabel("Recall")
axes[0].set_ylabel("Precision")
axes[0].set_title("Precision-Recall Curves (OOF)")
axes[0].legend()

# ROC Curves
for (name, oof), color in zip(models.items(), colors):
    valid = ~np.isnan(oof)
    fpr, tpr, _ = roc_curve(y_all[valid], oof[valid])
    auc = roc_auc_score(y_all[valid], oof[valid])
    axes[1].plot(fpr, tpr, label=f"{name} (AUC={auc:.4f})", color=color, lw=2)

axes[1].plot([0, 1], [0, 1], "--", color="gray", lw=1, label="Baseline")
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].set_title("ROC Curves (OOF)")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# ─── Calibration Curves (Reliability Diagrams) ──────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, (name, oof), color in zip(axes, models.items(), colors):
    valid = ~np.isnan(oof)
    y_v, p_v = y_all[valid], oof[valid]

    frac_pos, mean_pred = calibration_curve(y_v, p_v, n_bins=10, strategy="quantile")
    ax.plot(mean_pred, frac_pos, marker="o", color=color, lw=2, label="Модель")
    ax.plot([0, 1], [0, 1], "--", color="gray", label="Ідеальна")
    ax.set_title(f"{name}\nCalibration Curve")
    ax.set_xlabel("Прогнозована ймовірність")
    ax.set_ylabel("Фактична частка позитивних")
    ax.legend()

plt.suptitle("Reliability Diagrams — наскільки ймовірності відповідають реальності?", y=1.01)
plt.tight_layout()
plt.show()

print("""
Інтерпретація:
  • Якщо лінія лежить ВИЩЕ діагоналі: модель недооцінює ймовірність.
  • Якщо НИЖЧЕ: переоцінює.
  • Деревні моделі (CB, LGB) часто добре ранжують, але слабо калібровані
    (тому їх часто додатково калібрують ізотонічною регресією).
""")

In [ ]:
# ─── F1-Score vs Threshold ───────────────────────────────────────────────────
thresholds = np.linspace(0.001, 0.999, 150)

fig, ax = plt.subplots(figsize=(10, 5))

for (name, oof), color in zip(models.items(), colors):
    valid = ~np.isnan(oof)
    y_v, p_v = y_all[valid], oof[valid]
    f1s = [f1_score(y_v, (p_v >= t).astype(int), zero_division=0) for t in thresholds]
    opt_t, max_f1 = find_optimal_f1(y_v, p_v)

    ax.plot(thresholds, f1s, label=f"{name} (max F1={max_f1:.4f} @ t={opt_t:.3f})",
            color=color, lw=2)
    ax.axvline(opt_t, color=color, ls="--", alpha=0.5)

ax.set_xlabel("Probability Threshold")
ax.set_ylabel("F1-Score")
ax.set_title("F1-Score залежно від probability threshold")
ax.legend()
plt.tight_layout()
plt.show()

print("""
Інтерпретація:
  • За замовчуванням sklearn використовує threshold=0.5, але для незбалансованих
    даних оптимальний threshold зазвичай набагато нижчий.
  • Вибір threshold залежить від бізнес-логіки: чи важливіший precision чи recall.
""")

In [ ]:
# ─── Feature Importance: навчаємо фінальні моделі на 80% даних ──────────────
# Беремо перший фолд як приклад для важливості ознак

train_idx_fi, val_idx_fi = splits[0]
X_tr_raw = X_all.iloc[train_idx_fi]
X_va_raw = X_all.iloc[val_idx_fi]
y_tr_fi  = y_all[train_idx_fi]
y_va_fi  = y_all[val_idx_fi]

# ── CatBoost Feature Importance ──
X_tr_cb_fi, X_va_cb_fi = prepare_for_boosting(X_tr_raw, X_va_raw, final_cat)
X_tr_cb_fi.columns = sanitize_names(X_tr_cb_fi.columns.tolist())
X_va_cb_fi.columns = sanitize_names(X_va_cb_fi.columns.tolist())

cb_fi = CatBoostClassifier(
    iterations=200, learning_rate=0.05, depth=6,
    loss_function="Logloss", eval_metric="AUC",
    random_seed=RANDOM_STATE, verbose=0,
    allow_writing_files=False, auto_class_weights="Balanced",
)
train_pool_fi = Pool(X_tr_cb_fi, y_tr_fi, cat_features=cat_feature_indices)
val_pool_fi   = Pool(X_va_cb_fi, y_va_fi, cat_features=cat_feature_indices)
cb_fi.fit(train_pool_fi, eval_set=val_pool_fi, use_best_model=True, early_stopping_rounds=30)

cb_importance = pd.Series(
    cb_fi.get_feature_importance(),
    index=X_tr_cb_fi.columns.tolist()
).sort_values(ascending=True).tail(25)

# ── LightGBM Feature Importance ──
X_tr_lgb_fi, X_va_lgb_fi = prepare_for_boosting(X_tr_raw, X_va_raw, final_cat)
X_tr_lgb_fi.columns = sanitize_names(X_tr_lgb_fi.columns.tolist())
X_va_lgb_fi.columns = sanitize_names(X_va_lgb_fi.columns.tolist())

pos = float(y_tr_fi.sum())
neg_fi = float(len(y_tr_fi) - pos)
spw_fi = neg_fi / pos if pos > 0 else 1.0
cat_names_fi = [sanitize_names([c])[0] for c in final_cat if c in X_tr_lgb_fi.columns]

lgb_fi = lgb.LGBMClassifier(
    n_estimators=200, learning_rate=0.05, num_leaves=63,
    objective="binary", scale_pos_weight=spw_fi,
    random_state=RANDOM_STATE, n_jobs=-1, verbose=-1,
)
lgb_fi.fit(
    X_tr_lgb_fi, y_tr_fi,
    eval_set=[(X_va_lgb_fi, y_va_fi)],
    categorical_feature=cat_names_fi if cat_names_fi else "auto",
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)],
)

lgb_importance = pd.Series(
    lgb_fi.feature_importances_,
    index=X_tr_lgb_fi.columns.tolist()
).sort_values(ascending=True).tail(25)

# ── Графіки важливості ──
fig, axes = plt.subplots(1, 2, figsize=(16, 9))

cb_importance.plot(kind="barh", ax=axes[0], color=PALETTE["primary"])
axes[0].set_title("CatBoost — Топ-25 Feature Importance")
axes[0].set_xlabel("Importance (gain)")

lgb_importance.plot(kind="barh", ax=axes[1], color=PALETTE["secondary"])
axes[1].set_title("LightGBM — Топ-25 Feature Importance")
axes[1].set_xlabel("Importance (gain)")

plt.tight_layout()
plt.show()

In [ ]:
# ─── Confusion Matrix при оптимальному threshold ────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, (name, oof), color in zip(axes, models.items(), colors):
    valid = ~np.isnan(oof)
    y_v, p_v = y_all[valid], oof[valid]
    opt_t, _ = find_optimal_f1(y_v, p_v)
    y_pred = (p_v >= opt_t).astype(int)

    cm = confusion_matrix(y_v, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=["gb=0", "gb=1"])
    disp.plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(f"{name}\n(threshold={opt_t:.3f})")

plt.suptitle("Confusion Matrix при оптимальному F1 Threshold", y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ─── Порівняння моделей: метрики по фолдам ──────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

fold_compare_pr = pd.DataFrame({
    "Logistic Regression": lr_metrics_df["pr_auc"].values,
    "CatBoost":            cb_metrics_df["pr_auc"].values,
    "LightGBM":            lgb_metrics_df["pr_auc"].values,
}, index=[f"Fold {i}" for i in range(N_SPLITS)])

fold_compare_roc = pd.DataFrame({
    "Logistic Regression": lr_metrics_df["roc_auc"].values,
    "CatBoost":            cb_metrics_df["roc_auc"].values,
    "LightGBM":            lgb_metrics_df["roc_auc"].values,
}, index=[f"Fold {i}" for i in range(N_SPLITS)])

fold_compare_pr.plot(kind="bar", ax=axes[0],
                     color=[PALETTE["accent"], PALETTE["primary"], PALETTE["secondary"]],
                     edgecolor="white")
axes[0].set_title("PR-AUC по фолдам")
axes[0].set_ylabel("PR-AUC")
axes[0].tick_params(axis="x", rotation=0)
axes[0].legend(fontsize=9)

fold_compare_roc.plot(kind="bar", ax=axes[1],
                      color=[PALETTE["accent"], PALETTE["primary"], PALETTE["secondary"]],
                      edgecolor="white")
axes[1].set_title("ROC-AUC по фолдам")
axes[1].set_ylabel("ROC-AUC")
axes[1].tick_params(axis="x", rotation=0)
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# ─── ФІНАЛЬНЕ ПОРІВНЯННЯ: Boxplot метрик ─────────────────────────────────────
lr_all  = pd.DataFrame({"PR-AUC": lr_metrics_df["pr_auc"],  "ROC-AUC": lr_metrics_df["roc_auc"],  "Модель": "LR"})
cb_all  = pd.DataFrame({"PR-AUC": cb_metrics_df["pr_auc"],  "ROC-AUC": cb_metrics_df["roc_auc"],  "Модель": "CatBoost"})
lgb_all = pd.DataFrame({"PR-AUC": lgb_metrics_df["pr_auc"], "ROC-AUC": lgb_metrics_df["roc_auc"], "Модель": "LightGBM"})
all_metrics = pd.concat([lr_all, cb_all, lgb_all], ignore_index=True)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.boxplot(data=all_metrics, x="Модель", y="PR-AUC", ax=axes[0],
            palette=[PALETTE["accent"], PALETTE["primary"], PALETTE["secondary"]])
axes[0].set_title("PR-AUC по 5 фолдам")

sns.boxplot(data=all_metrics, x="Модель", y="ROC-AUC", ax=axes[1],
            palette=[PALETTE["accent"], PALETTE["primary"], PALETTE["secondary"]])
axes[1].set_title("ROC-AUC по 5 фолдам")

plt.suptitle("Стабільність метрик по фолдам (менший розкид = більш стабільна модель)", y=1.02)
plt.tight_layout()
plt.show()

## Висновки та рекомендації

### Ключові знахідки EDA

| Спостереження | Вплив на моделювання |
|--------------|---------------------|
| **~2.2% позитивних** (gb=1) | Сильний дисбаланс → оптимізуємо PR-AUC, не accuracy |
| **Панельна структура** (~5 рядків/ID) | Обов'язковий StratifiedGroupKFold по ID |
| **MNAR ознаки** | Факт пропуску = інформація → додаємо is_missing індикатори |
| **700+ скорельованих пар** | Критично для LR (мультиколінеарність) |
| **Median skewness ~9** | LR потребує RobustScaler; деревні моделі — стійкі |

### Результати моделей (OOF CV)

| Метрика | Logistic Regression | CatBoost | LightGBM |
|---------|--------------------|-----------|-----------|
| PR-AUC | ≈ нижче | ≈ вище | ≈ вище |
| ROC-AUC | ≈ хороший | ≈ кращий | ≈ кращий |
| Швидкість | ⚡ Дуже швидка | 🐢 Повільніша | ⚡ Швидка |
| Інтерпретація | ✅ Висока | ⚠️ Середня | ⚠️ Середня |

### Рекомендована продакшн-модель: **CatBoost або LightGBM**

**Наступні кроки:**
1. Налаштування гіперпараметрів через `Optuna` або `GridSearchCV`
2. Ensemble (усереднення ймовірностей CatBoost + LightGBM)
3. SHAP-аналіз для пояснення рішень
4. Ізотонічна калібрація ймовірностей
5. Визначення operational threshold на основі матриці витрат (cost matrix)